# NB07 — LoRA Under the Hood

**North star callback:** Unsloth's LoRA is faster than PEFT's because it fuses the
adapter computation into the base layer's forward pass — eliminating the overhead of
PEFT's hook-based adapter insertion.

LoRA: instead of updating W (d×k), train two small matrices A (d×r) and B (r×k),
where r << min(d,k). The adapted output is `xW + x(AB)`.

## 1. LoRA math from scratch

In [1]:
import math
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """Minimal LoRA wrapper around a frozen linear layer."""
    def __init__(self, base: nn.Linear, r: int = 16, alpha: float = 32.0):
        super().__init__()
        d_in, d_out = base.in_features, base.out_features
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)

        self.lora_A = nn.Parameter(torch.randn(d_in, r) / math.sqrt(r))
        self.lora_B = nn.Parameter(torch.zeros(r, d_out))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.lora_A @ self.lora_B) * self.scale


linear = nn.Linear(4096, 4096, bias=False, device="cuda", dtype=torch.bfloat16)
lora_linear = LoRALinear(linear, r=16, alpha=32).cuda().bfloat16()

x = torch.randn(2, 512, 4096, device="cuda", dtype=torch.bfloat16)
out = lora_linear(x)
print(f"Output shape: {out.shape}")
trainable = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)
total     = sum(p.numel() for p in lora_linear.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

Output shape: torch.Size([2, 512, 4096])
Trainable params: 131,072 / 16,908,288  (0.78%)


## 2. Hook-based LoRA vs fused forward\n\nPEFT wires adapters via `register_forward_hook` — an extra Python call dispatched after\nevery layer's `forward()`. Our `LoRALinear` computes base + adapter inline in one pass.\nBelow we replicate PEFT's approach manually to isolate the hook overhead.

In [2]:
import sys
sys.path.insert(0, "..")
from utils.benchmark import compare

# Replicate PEFT's hook mechanism manually
# (get_peft_model requires torchao >= 0.16.0; we have 0.8.0)
base_hooked = nn.Linear(4096, 4096, bias=False, device="cuda", dtype=torch.bfloat16)
for p in base_hooked.parameters():
    p.requires_grad_(False)

hook_A = nn.Parameter(torch.randn(4096, 16, device="cuda", dtype=torch.bfloat16) / math.sqrt(16))
hook_B = nn.Parameter(torch.zeros(16, 4096, device="cuda", dtype=torch.bfloat16))
hook_scale = 32.0 / 16

base_hooked.weight.data.copy_(linear.weight.data)

def lora_forward_hook(module, inp, output):
    """Simulates PEFT's LoRA hook: runs after the base linear forward."""
    return output + (inp[0] @ hook_A @ hook_B) * hook_scale

hook_handle = base_hooked.register_forward_hook(lora_forward_hook)

results = compare(
    fns={
        "frozen_base": lambda: linear(x),
        "hook_lora":   lambda: base_hooked(x),
        "fused_lora":  lambda: lora_linear(x),
    },
    notebook="nb07",
    experiment="lora_forward",
    n_warmup=10, n_repeat=50,
)
for label, r in results.items():
    print(f"{label:12s}: {r.latency_ms:.3f} ms")

frozen_base : 0.215 ms
hook_lora   : 0.243 ms
fused_lora  : 0.243 ms


## 3. QLoRA: 4-bit base + bf16 adapters

QLoRA keeps the base model in 4-bit NF4 (loaded via bitsandbytes).
The adapter matrices A and B remain in bf16.
On the forward pass: dequantize the base weight → compute xW → add xAB.
The dequantization is the bottleneck; NB08 addresses this with a custom kernel.

## 4. Read Unsloth's LoRA implementation

Unsloth's fused LoRA is in `unsloth/kernels/fast_lora.py`.
Look for: `apply_lora_mlp`, `apply_lora_qkv` — these compute base + adapter
in one fused kernel call rather than two separate matmuls.

In [3]:
sys.path.insert(0, "../unsloth")
from unsloth.kernels import fast_lora
import inspect
src = inspect.getsource(fast_lora)
print(src[:3000])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Copyright 2023-present Daniel Han-Chen & the Unsloth team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import torch
from .utils import (
    _maybe_fake_quantize_activations,
    fast_dequantize,
    QUANT_STATE,
    get_lora_parameters,
    get_lora_parameters_bias,
    matmul_lora,
    torch_amp_custom_fwd,
    torch_amp_custom_bwd,
)


class LoRA_MLP(torch.autograd.Function):
    """
    ### LoRA weights
    G = G + Ag @ Bg
    U = U + Au @ Bu
    W = W + Aw @ Bw

  

## 5. Exercises

1. **Implement LoRA merge**: add a `merge_weights()` method to `LoRALinear` that
   absorbs A and B into the base weight (`W += scale * A @ B`) and removes the adapters.
   Verify that output is unchanged after merging.

2. **Multi-rank experiment**: benchmark `LoRALinear` with r=4, 8, 16, 32, 64.
   Plot latency vs r. At what r does LoRA overhead become significant?

3. **Understand the PEFT hook overhead**: use `torch.profiler` to trace a forward
   pass through `peft_model`. Where does the extra time go compared to `fused_lora`?